In [15]:
import pandas as pd
import numpy as np
import os

In [16]:
OUTPUT_POSTPROCESSING_DIR_PATH = os.getcwd()
SSP_MODELING_DIR_PATH = os.path.dirname(OUTPUT_POSTPROCESSING_DIR_PATH)
SSP_OUTPUT_DIR_PATH = os.path.join(SSP_MODELING_DIR_PATH, "ssp_run_output")
CW_DATA_DIR_PATH = os.path.join(OUTPUT_POSTPROCESSING_DIR_PATH, "data")

In [17]:
ISO3 = "BGR"
REGION_NAME = "bulgaria"
RUN_DIR_PATH = os.path.join(SSP_OUTPUT_DIR_PATH, "sisepuede_results_sisepuede_run_2025-10-03T11;35;32.757306")

### Load emission targets and ssp outputs dfs

In [18]:
# Load emission targets
emission_targets_df = pd.read_csv(os.path.join(CW_DATA_DIR_PATH, "emission_targets_bulgaria_2022.csv"))
emission_targets_df.head()

,Subsector,Gas,Edgar_Class,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,ids,BGR
0,agrc,ch4,AG - Crops:CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,1:agrc:ch4,0.101736
1,agrc,co2,AG - Crops:CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_agrc_biomass_bevs_and_spices...,2:agrc:co2,1.071093
2,agrc,n2o,AG - Crops:N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_agrc_biomass_burning:emissio...,3:agrc:n2o,3.236068
3,lvst,ch4,AG - Livestock:CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,4:lvst:ch4,1.068538
4,lsmm,ch4,AG - Livestock:CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,5:lsmm:ch4,1.068538


In [19]:
# Load output data
ssp_output_df = pd.read_csv(os.path.join(RUN_DIR_PATH, 
                                         "sisepuede_results_sisepuede_run_2025-10-03T11;35;32.757306_WIDE_INPUTS_OUTPUTS.csv"))
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,bulgaria,0,0.0,1.885187e+06,4530.325442,77118.917398,40648.867124,7620.499506,2.386984e+06,...,12.059233,2.946800,6.177415,0.0,2.755621,8.636027,73.140598,37.547100,28.821448,92.81
1,0,bulgaria,1,0.0,1.885174e+06,4530.296332,77118.421867,40648.605933,7620.450540,2.386969e+06,...,12.539800,3.455483,7.165802,0.0,3.259699,8.228317,71.163825,39.000100,29.765171,92.81
2,0,bulgaria,2,0.0,1.893163e+06,4549.493932,77445.219169,40820.858612,7652.742989,2.397084e+06,...,12.161233,3.041817,11.119348,0.0,3.958685,8.302446,89.448975,39.119475,28.720595,92.81
3,0,bulgaria,3,0.0,1.886444e+06,4533.346613,77170.346246,40675.974927,7625.581443,2.388576e+06,...,12.289167,3.375283,5.436126,0.0,4.167037,8.846059,90.190264,37.352525,29.482348,92.81
4,0,bulgaria,4,0.0,1.869713e+06,4493.139884,76485.914305,40315.215415,7557.949357,2.367392e+06,...,12.337700,2.992183,6.177415,0.0,3.226093,7.882382,84.012849,38.551850,32.769776,92.81


### Obtain the ssp output values in the emission targets format

In [20]:
def sum_vars_from_ssp_outputs(
    emission_targets_df: pd.DataFrame,
    ssp_outputs_df: pd.DataFrame,
    vars_col: str = "Vars",
    out_col: str = "ssp_total",
    record_missing_col: str | None = "missing_vars",
    ssp_filter: dict | None = None,
) -> pd.DataFrame:
    """
    For each row in emission_targets_df, split the colon-separated strings in `vars_col`,
    find those columns in ssp_outputs_df, sum their values (over rows & columns), and
    write the total to `out_col` in emission_targets_df.

    Parameters
    ----------
    emission_targets_df : DataFrame
        Must contain a string column `vars_col` with colon-separated names.
    ssp_outputs_df : DataFrame
        Wide table whose columns include the names referenced by `emission_targets_df[vars_col]`.
    vars_col : str
        Column in emission_targets_df with colon-separated variable names.
    out_col : str
        New column to create in emission_targets_df with totals from ssp_outputs_df.
    record_missing_col : str | None
        If provided, creates a column listing any missing vars for each row.
    df2_filter : dict | None
        Optional filters to reduce ssp_outputs_df before summing, e.g.
        {"region": "egypt", "time_period": 7}

    Returns
    -------
    DataFrame
        emission_targets_df with new column `out_col` (and `record_missing_col` if requested).
    """
    # Optionally filter ssp_outputs_df by key=value pairs (e.g., region/time_period)
    if ssp_filter:
        mask = pd.Series(True, index=ssp_outputs_df.index)
        for k, v in ssp_filter.items():
            mask &= (ssp_outputs_df[k] == v)
        ssp_view = ssp_outputs_df.loc[mask]
    else:
        ssp_view = ssp_outputs_df

    # Ensure we only operate on numeric data when summing
    numeric_cols = set(ssp_view.select_dtypes(include=[np.number]).columns)

    def _total_for_vars(vars_str: str):
        if pd.isna(vars_str) or not str(vars_str).strip():
            return np.nan, []

        # Split, strip, and deduplicate while preserving order
        raw = [s.strip() for s in str(vars_str).split(":") if s.strip()]
        seen = set()
        cols = [c for c in raw if not (c in seen or seen.add(c))]

        present = [c for c in cols if c in ssp_view.columns and c in numeric_cols]
        missing = [c for c in cols if c not in ssp_view.columns or c not in numeric_cols]

        if not present or ssp_view.empty:
            return np.nan, missing

        # Sum over all filtered rows & all present columns
        vals = ssp_view[present].to_numpy(dtype=float, copy=False)
        total = np.nansum(vals)
        return float(total), missing

    totals, missings = [], []
    for v in emission_targets_df[vars_col].astype("string"):
        total, missing = _total_for_vars(v)
        totals.append(total)
        missings.append(missing)

    emission_targets_df = emission_targets_df.copy()
    emission_targets_df[out_col] = totals
    if record_missing_col is not None:
        emission_targets_df[record_missing_col] = missings

    return emission_targets_df


# -----------------------------
# Example usage
# -----------------------------

# If DF2 has a single row for the target (e.g., region="egypt", a specific time_period):
# df2_filter = {"region": "egypt"}              # or {"region": "egypt", "time_period": 7}
# If you want to sum across all rows of DF2, set df2_filter = None.

# df1_result = sum_vars_from_df2(DF1, DF2, vars_col="Vars",
#                                out_col="DF2_total",
#                                record_missing_col="Missing_in_DF2",
#                                df2_filter={"region": "egypt"})
# print(df1_result.head())


In [21]:
ssp_output_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,0,bulgaria,0,0.0,1.885187e+06,4530.325442,77118.917398,40648.867124,7620.499506,2.386984e+06,...,12.059233,2.946800,6.177415,0.0,2.755621,8.636027,73.140598,37.547100,28.821448,92.81
1,0,bulgaria,1,0.0,1.885174e+06,4530.296332,77118.421867,40648.605933,7620.450540,2.386969e+06,...,12.539800,3.455483,7.165802,0.0,3.259699,8.228317,71.163825,39.000100,29.765171,92.81
2,0,bulgaria,2,0.0,1.893163e+06,4549.493932,77445.219169,40820.858612,7652.742989,2.397084e+06,...,12.161233,3.041817,11.119348,0.0,3.958685,8.302446,89.448975,39.119475,28.720595,92.81
3,0,bulgaria,3,0.0,1.886444e+06,4533.346613,77170.346246,40675.974927,7625.581443,2.388576e+06,...,12.289167,3.375283,5.436126,0.0,4.167037,8.846059,90.190264,37.352525,29.482348,92.81
4,0,bulgaria,4,0.0,1.869713e+06,4493.139884,76485.914305,40315.215415,7557.949357,2.367392e+06,...,12.337700,2.992183,6.177415,0.0,3.226093,7.882382,84.012849,38.551850,32.769776,92.81


In [22]:
emission_targets_df_extended = sum_vars_from_ssp_outputs(emission_targets_df, ssp_output_df, vars_col="Vars",
                               out_col="ssp_emission",
                               record_missing_col="missing_in_ssp_outputs",
                               ssp_filter={"region": REGION_NAME, "primary_id": 0, "time_period": 7})

emission_targets_df_extended.head()

,Subsector,Gas,Edgar_Class,Edgar_Sector,Edgar_Subsector,Edgar_Subsector_Synthetic,Vars,ids,BGR,ssp_emission,missing_in_ssp_outputs
0,agrc,ch4,AG - Crops:CH4,Agriculture,AG - Crops,AG - Crops,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,1:agrc:ch4,0.101736,0.139853,[]
1,agrc,co2,AG - Crops:CO2,Agriculture,AG - Crops,AG - Crops,emission_co2e_co2_agrc_biomass_bevs_and_spices...,2:agrc:co2,1.071093,0.285394,[]
2,agrc,n2o,AG - Crops:N2O,Agriculture,AG - Crops,AG - Crops,emission_co2e_n2o_agrc_biomass_burning:emissio...,3:agrc:n2o,3.236068,1.035653,[]
3,lvst,ch4,AG - Livestock:CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,4:lvst:ch4,1.068538,1.375970,[]
4,lsmm,ch4,AG - Livestock:CH4,Agriculture,AG - Livestock,AG - Livestock,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,5:lsmm:ch4,1.068538,0.661047,[]


### Create diff report

In [23]:
# subset the emission targets to create the diff report template
diff_report_df = emission_targets_df_extended[[
    "Subsector",
    "Edgar_Class",
    ISO3,
    "ssp_emission",
]].copy()
diff_report_df.head()

,Subsector,Edgar_Class,BGR,ssp_emission
0,agrc,AG - Crops:CH4,0.101736,0.139853
1,agrc,AG - Crops:CO2,1.071093,0.285394
2,agrc,AG - Crops:N2O,3.236068,1.035653
3,lvst,AG - Livestock:CH4,1.068538,1.375970
4,lsmm,AG - Livestock:CH4,1.068538,0.661047


In [24]:
# merge subsector an id into a single column for clarity
diff_report_df["subsector_id"] = diff_report_df["Subsector"] + " - " + diff_report_df["Edgar_Class"]
diff_report_df = diff_report_df.drop(columns=["Subsector", "Edgar_Class"])
diff_report_df.head()

,BGR,ssp_emission,subsector_id
0,0.101736,0.139853,agrc - AG - Crops:CH4
1,1.071093,0.285394,agrc - AG - Crops:CO2
2,3.236068,1.035653,agrc - AG - Crops:N2O
3,1.068538,1.375970,lvst - AG - Livestock:CH4
4,1.068538,0.661047,lsmm - AG - Livestock:CH4


In [25]:
#rename region column
diff_report_df = diff_report_df.rename(columns={ISO3: "inventory_emission"})
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id
0,0.101736,0.139853,agrc - AG - Crops:CH4
1,1.071093,0.285394,agrc - AG - Crops:CO2
2,3.236068,1.035653,agrc - AG - Crops:N2O
3,1.068538,1.375970,lvst - AG - Livestock:CH4
4,1.068538,0.661047,lsmm - AG - Livestock:CH4


In [26]:
# Create inventory_share column
diff_report_df["inventory_share"] = diff_report_df["inventory_emission"] / diff_report_df["inventory_emission"].sum()
diff_report_df

,inventory_emission,ssp_emission,subsector_id,inventory_share
0,0.101736,0.139853,agrc - AG - Crops:CH4,0.001739
1,1.071093,0.285394,agrc - AG - Crops:CO2,0.018311
2,3.236068,1.035653,agrc - AG - Crops:N2O,0.055322
3,1.068538,1.375970,lvst - AG - Livestock:CH4,0.018267
4,1.068538,0.661047,lsmm - AG - Livestock:CH4,0.018267
...,...,...,...,...
69,5.148576,0.066318,waso - Waste - Solid Waste:CH4,0.088017
70,0.006775,1.323224,waso - Waste - Solid Waste:CO2,0.000116
71,0.009935,0.010321,waso - Waste - Solid Waste:N2O,0.000170
72,1.130880,0.794170,trww - Waste - Wastewater Treatment:CH4,0.019333


In [27]:
# Calculate error column, avoid division by zero by adding a small constant to the denominator
epsilon = 1e-8
diff_report_df["error"] = (diff_report_df["ssp_emission"] - diff_report_df["inventory_emission"]).abs() / (diff_report_df["inventory_emission"] + epsilon)
diff_report_df["squared_error"] = (diff_report_df["error"] ** 2)
diff_report_df.head()

,inventory_emission,ssp_emission,subsector_id,inventory_share,error,squared_error
0,0.101736,0.139853,agrc - AG - Crops:CH4,0.001739,0.374668,0.140376
1,1.071093,0.285394,agrc - AG - Crops:CO2,0.018311,0.733548,0.538093
2,3.236068,1.035653,agrc - AG - Crops:N2O,0.055322,0.679966,0.462353
3,1.068538,1.375970,lvst - AG - Livestock:CH4,0.018267,0.287713,0.082779
4,1.068538,0.661047,lsmm - AG - Livestock:CH4,0.018267,0.381354,0.145431


In [28]:
# Set subsector_id at the beginning of the df
diff_report_df = diff_report_df[[
    "subsector_id",
    "inventory_emission",
    "ssp_emission",
    "inventory_share",
    "error",
    "squared_error"
]]

# sort by squared_error descending
diff_report_df = diff_report_df.sort_values(by="squared_error", ascending=False)
diff_report_df.head(10)

,subsector_id,inventory_emission,ssp_emission,inventory_share,error,squared_error
62,frst - LULUCF - Forest Land Removals:CO2,0.000000,39.728842,0.000000,3.972884e+09,1.578381e+19
64,frst - LULUCF - HWP:CO2,0.000000,-36.341240,0.000000,3.634124e+09,1.320686e+19
66,soil - LULUCF - Organic Soil:N2O,0.000000,16.273066,0.000000,1.627307e+09,2.648127e+18
61,frst - LULUCF - Forest Land:CH4,0.000000,0.110324,0.000000,1.103239e+07,1.217136e+14
59,lndu - LULUCF - Deforestation:CH4,0.000000,0.091378,0.000000,9.137757e+06,8.349860e+13
54,ippu - IN - Industrial Processes:HFC,0.001386,1.709831,0.000024,1.232691e+03,1.519526e+06
50,ippu - IN - Industrial Processes:HFC,0.001386,0.613238,0.000024,4.414683e+02,1.948942e+05
29,ippu - IN - Industrial Processes:HFC,0.001386,0.604241,0.000024,4.349766e+02,1.892046e+05
28,ippu - IN - Industrial Processes:HFC,0.001386,0.398555,0.000024,2.865686e+02,8.212157e+04
70,waso - Waste - Solid Waste:CO2,0.006775,1.323224,0.000116,1.943019e+02,3.775322e+04


In [29]:
diff_report_df.tail(40)

,subsector_id,inventory_emission,ssp_emission,inventory_share,error,squared_error
46,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
45,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
44,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
56,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
41,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
37,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
39,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
38,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
36,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986
35,ippu - IN - Industrial Processes:HFC,0.001386,0.000000,0.000024,0.999993,0.999986


### Save diff table

In [30]:
diff_report_df.to_csv(os.path.join(RUN_DIR_PATH, f"diff_report_{REGION_NAME}.csv"), index=False)